# 03. Canonical language/vision/generative Transformer references

This notebook keeps GPT-2 Small, ViT-B/16, and DiT-B/2 structural topology while reducing only tensor scale. π0 is implemented once, in notebook 23, to avoid maintaining a duplicate robotics implementation.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")


## 1. GPT-2 Small — 12 decoder blocks / 12 heads / learned absolute positions


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        batch, length, dim = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(batch, length, dim)
        return self.out(y)


class GPT2Block(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class SmallWidthGPT2(nn.Module):
    def __init__(self, vocab=128, max_len=32, dim=48, depth=12, heads=12):
        super().__init__()
        self.token = nn.Embedding(vocab, dim)
        self.position = nn.Embedding(max_len, dim)
        self.blocks = nn.ModuleList([GPT2Block(dim, heads) for _ in range(depth)])
        self.final_norm = nn.LayerNorm(dim)

    def forward(self, token_ids):
        length = token_ids.size(1)
        positions = torch.arange(length, device=token_ids.device)
        x = self.token(token_ids) + self.position(positions)[None]
        for block in self.blocks:
            x = block(x)
        return F.linear(self.final_norm(x), self.token.weight)


gpt = SmallWidthGPT2().to(device)
assert len(gpt.blocks) == 12
assert gpt.blocks[0].attn.heads == 12
ids = torch.randint(0, 128, (1, 6), device=device)
gpt(ids).square().mean().backward()


## 2. ViT-B/16 — 12 encoder blocks / 12 heads / learned absolute positions


In [ ]:
class ViTBlock(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        q = self.norm1(x)
        y, _ = self.attn(q, q, q, need_weights=False)
        x = x + y
        return x + self.mlp(self.norm2(x))


class SmallWidthViTB16(nn.Module):
    def __init__(self, image_size=32, dim=48, depth=12, heads=12, classes=10):
        super().__init__()
        patch_size = 16
        self.patch_size = patch_size
        self.patch = nn.Conv2d(3, dim, patch_size, stride=patch_size)
        patch_count = (image_size // patch_size) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.randn(1, patch_count + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([ViTBlock(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, image):
        x = self.patch(image).flatten(2).transpose(1, 2)
        cls = self.cls.expand(image.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.position[:, : x.size(1)]
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x[:, 0]))


vit = SmallWidthViTB16().to(device)
assert len(vit.blocks) == 12
assert vit.patch_size == 16
vit(torch.randn(1, 3, 32, 32)).square().mean().backward()


## 3. DiT-B/2 — fixed 2D sin-cos position embedding, adaLN-Zero, 12 blocks / 12 heads


In [ ]:
def sincos_1d(dim, positions):
    assert dim % 2 == 0
    omega = torch.arange(dim // 2, dtype=torch.float32, device=positions.device)
    omega = 1.0 / (10000 ** (omega / (dim / 2)))
    angles = positions.reshape(-1, 1).float() * omega.reshape(1, -1)
    return torch.cat([angles.sin(), angles.cos()], dim=1)


def sincos_2d(dim, grid_size, device):
    assert dim % 4 == 0
    y, x = torch.meshgrid(
        torch.arange(grid_size, device=device),
        torch.arange(grid_size, device=device),
        indexing="ij",
    )
    emb_x = sincos_1d(dim // 2, x.reshape(-1))
    emb_y = sincos_1d(dim // 2, y.reshape(-1))
    return torch.cat([emb_x, emb_y], dim=1)[None]


def timestep_embedding(t, dim):
    half = dim // 2
    freq = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=t.device, dtype=torch.float32)
        / half
    )
    args = t.float()[:, None] * freq[None]
    return torch.cat([args.cos(), args.sin()], dim=-1)


def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.heads = heads
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.ada[-1].weight)
        nn.init.zeros_(self.ada[-1].bias)

    def forward(self, x, condition):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.ada(condition).chunk(6, -1)
        q = modulate(self.norm1(x), shift_a, scale_a)
        y, _ = self.attn(q, q, q, need_weights=False)
        x = x + gate_a[:, None] * y
        z = modulate(self.norm2(x), shift_m, scale_m)
        return x + gate_m[:, None] * self.mlp(z)


class DiTFinal(nn.Module):
    def __init__(self, dim, patch_size, out_channels):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.out = nn.Linear(dim, patch_size * patch_size * out_channels)
        nn.init.zeros_(self.ada[-1].weight)
        nn.init.zeros_(self.ada[-1].bias)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x, condition):
        shift, scale = self.ada(condition).chunk(2, -1)
        return self.out(modulate(self.norm(x), shift, scale))


class SmallWidthDiTB2(nn.Module):
    def __init__(self, image_size=8, channels=4, dim=48, depth=12, heads=12, classes=10):
        super().__init__()
        self.image_size = image_size
        self.patch_size = 2
        self.channels = channels
        self.patch = nn.Conv2d(channels, dim, 2, stride=2)
        grid = image_size // 2
        position = sincos_2d(dim, grid, torch.device("cpu"))
        self.register_buffer("position", position, persistent=False)
        self.time_mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )
        self.label = nn.Embedding(classes + 1, dim)
        self.blocks = nn.ModuleList([DiTBlock(dim, heads) for _ in range(depth)])
        self.final = DiTFinal(dim, 2, channels * 2)

    def forward(self, image, t, label):
        x = self.patch(image).flatten(2).transpose(1, 2)
        x = x + self.position.to(x.device, x.dtype)
        condition = self.time_mlp(timestep_embedding(t, x.size(-1))) + self.label(label)
        for block in self.blocks:
            x = block(x, condition)
        patches = self.final(x, condition).transpose(1, 2)
        return F.fold(
            patches,
            output_size=(self.image_size, self.image_size),
            kernel_size=2,
            stride=2,
        )


dit = SmallWidthDiTB2().to(device)
assert len(dit.blocks) == 12
assert all(block.heads == 12 for block in dit.blocks)
assert not isinstance(dit.position, nn.Parameter)
latent = torch.randn(1, 4, 8, 8)
t = torch.tensor([500])
y = torch.tensor([3])
dit(latent, t, y).square().mean().backward()


## Audit result

GPT-2 and ViT retain their reference block counts and positional-embedding types. DiT-B/2 uses fixed 2D sin-cos positional encoding, 12 blocks, 12 heads, and adaLN-Zero.
